# 04 — Evaluation, calibration and explainability (Member 4)

This notebook reads outputs from `scripts/run_evaluation_explainability.py`; it does not fit or select models. All reported results are validation-only. The held-out test partition remains locked.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
TABLES = ROOT / 'results' / 'tables' / 'member4'
FIGURES = ROOT / 'results' / 'figures' / 'member4'
FIGURES.mkdir(parents=True, exist_ok=True)
summary = pd.read_csv(TABLES / 'evaluation_summary.csv')
selected = summary[summary['selected_for_model']].copy()
selected['candidate_name'] = np.where(
    selected['calibration_method'].eq('uncalibrated'),
    selected['model_name'],
    selected['model_name'] + '__' + selected['calibration_method'],
)
display(summary.round(4))
print('Selected by validation Brier score, then log loss:')
display(selected[['model_name', 'calibration_method', 'brier_score', 'log_loss',
                  'expected_calibration_error', 'roc_auc',
                  'pr_auc_average_precision']].round(4))

## Discrimination
ROC-AUC and average precision assess ranking. Curves below use exactly the same validation observations, verified through the shared prediction schema.

In [ ]:
predictions = {}
for row in selected.itertuples():
    frame = pd.read_csv(TABLES / f'{row.candidate_name}_validation_predictions.csv')
    assert frame['split'].eq('validation').all()
    predictions[row.candidate_name] = frame.sort_values('sample_id')
reference = next(iter(predictions.values()))[['sample_id', 'y_true']]
for name, frame in predictions.items():
    pd.testing.assert_frame_equal(frame[['sample_id', 'y_true']].reset_index(drop=True),
                                  reference.reset_index(drop=True))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, frame in predictions.items():
    RocCurveDisplay.from_predictions(frame['y_true'], frame['y_pred_proba'], name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(frame['y_true'], frame['y_pred_proba'], name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], 'k--', alpha=.4)
axes[1].axhline(reference['y_true'].mean(), color='k', linestyle='--', alpha=.4)
axes[0].set_title('ROC — validation')
axes[1].set_title('Precision–recall — validation')
for ax in axes: ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES / 'roc_pr_validation.png', dpi=160, bbox_inches='tight'); plt.show()

## Decision metrics at the fixed reporting threshold
These confusion matrices use threshold 0.5 for consistent reporting, not as a cost-optimal decision rule. Threshold selection belongs to Member 5.

In [ ]:
fig, axes = plt.subplots(1, len(predictions), figsize=(5 * len(predictions), 4), squeeze=False)
for ax, (name, frame) in zip(axes.flat, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        frame['y_true'], (frame['y_pred_proba'] >= .5).astype(int),
        labels=[0, 1], display_labels=['No delinquency', 'Delinquency'],
        cmap='Blues', colorbar=False, ax=ax,
    )
    ax.set_title(name)
plt.tight_layout(); plt.savefig(FIGURES / 'confusion_matrices_validation.png', dpi=160, bbox_inches='tight'); plt.show()

## Probability calibration
Sigmoid and isotonic calibration are fit only through stratified training folds. The selected candidate minimizes validation Brier score, with log loss as a deterministic tie-break. ECE/MCE are descriptive and depend on binning.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for row in selected.itertuples():
    bins = pd.read_csv(TABLES / f'{row.candidate_name}_calibration_bins.csv')
    ax.plot(bins['mean_predicted_probability'], bins['observed_positive_rate'],
            marker='o', label=row.candidate_name)
ax.plot([0, 1], [0, 1], 'k--', label='ideal')
ax.set(xlabel='Mean predicted probability', ylabel='Observed positive rate',
       title='Reliability diagram — validation')
ax.grid(alpha=.25); ax.legend(); plt.tight_layout(); plt.savefig(FIGURES / 'reliability_validation.png', dpi=160, bbox_inches='tight'); plt.show()

## Uncertainty and duplicate sensitivity
Percentile intervals use 1,000 class-stratified bootstrap resamples. Pairwise deltas reuse the same resampled observations. The duplicate sensitivity table keeps the approved split unchanged and reports metrics after excluding validation rows that exactly match a training row.

In [ ]:
intervals = pd.read_csv(TABLES / 'bootstrap_confidence_intervals.csv')
pairwise = pd.read_csv(TABLES / 'paired_bootstrap_differences.csv')
duplicate_sensitivity = pd.read_csv(TABLES / 'duplicate_sensitivity.csv')
display(intervals.round(4))
display(pairwise.round(4))
display(duplicate_sensitivity.round(4))

## Global SHAP importance
Mean absolute SHAP magnitude describes how strongly the selected XGBoost ranking model uses each transformed feature. It does not establish causality.

In [ ]:
importance = pd.read_csv(TABLES / 'shap_global_importance.csv').head(15)
display(importance.round(5))
importance.sort_values('mean_absolute_shap').plot.barh(
    x='feature', y='mean_absolute_shap', legend=False, figsize=(9, 6),
    title='Global SHAP importance — validation sample')
plt.xlabel('Mean |SHAP value|'); plt.tight_layout(); plt.savefig(FIGURES / 'shap_global_importance.png', dpi=160, bbox_inches='tight'); plt.show()

## Representative local explanations
Cases are chosen deterministically from validation: a high-risk positive, a false positive and a false negative when available. Feature contributions explain the model score, not the reason a borrower became delinquent.

In [ ]:
cases = pd.read_csv(TABLES / 'shap_representative_cases.csv')
local = pd.read_csv(TABLES / 'shap_local_explanations.csv')
display(cases)
fig, axes = plt.subplots(len(cases), 1, figsize=(10, 4 * len(cases)), squeeze=False)
for ax, case in zip(axes.flat, cases.itertuples()):
    values = local[local['sample_id'].eq(case.sample_id)].sort_values('absolute_shap')
    colors = np.where(values['shap_value'] >= 0, 'tab:red', 'tab:blue')
    ax.barh(values['feature'], values['shap_value'], color=colors)
    ax.set_title(f'{case.case_type}: sample {case.sample_id}, y={case.y_true}, p={case.y_pred_proba:.3f}')
    ax.set_xlabel('SHAP contribution to model output'); ax.grid(axis='x', alpha=.2)
plt.tight_layout(); plt.savefig(FIGURES / 'shap_local_cases.png', dpi=160, bbox_inches='tight'); plt.show()

## Handoff and limitations
- Member 5 must choose cost-sensitive thresholds from validation predictions only and state the `C_FN:C_FP` orientation.
- Member 1 performs one final locked test evaluation only after model, calibration and threshold decisions are approved.
- Bootstrap intervals quantify sampling uncertainty conditional on this fixed validation split; they do not replace external or temporal validation.
- Duplicate sensitivity is descriptive and does not authorize changing the approved shared split.
- Calibration and SHAP outputs are development evidence, not production or causal claims.